In [1]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('Data/vehicles.csv')

In [2]:
# Filter VINs by real VIN format:
# - exactly 17 characters
# - uppercase letters/digits only
# - excludes I, O, Q
vin_series = df['VIN'].astype(str).str.strip().str.upper()
df = df[vin_series.str.fullmatch(r'[A-HJ-NPR-Z0-9]{17}', na=False)]

VIN_sample = df["VIN"].head(5).tolist()

In [3]:
import requests

# NHTSA vPIC(Vehicle Product Information Catalog) API
# Don't Need API KEY
API_URL = "https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVINValuesBatch/"

payload = {"format": "json", "data": ";".join(VIN_sample)}
response = requests.post(API_URL, data=payload, timeout=20)
response.raise_for_status()

decoded_rows = response.json().get("Results", [])
decoded_df = pd.DataFrame(decoded_rows)

print(f"Requested VIN count: {len(VIN_sample)}")
print(decoded_df[["VIN", "ModelYear", "Make", "Model", "Series", "FuelTypePrimary", "Trim", "DriveType", "EngineCylinders", "DisplacementL", "TransmissionStyle", "Doors"]].head())

# 문자열 표준화 필요
# 결측 처리 규칙 필요
# VIN으로 현재 DataSet이 올바른지 확인

Requested VIN count: 5
                 VIN ModelYear       Make      Model  \
0  3GTP1VEC4EG551563      2014        GMC     Sierra   
1  1GCSCSE06AZ123805      2010  CHEVROLET  Silverado   
2  3GCPWCED5LG130317      2020  CHEVROLET  Silverado   
3  5TFRM5F17HX120972      2017     TOYOTA     Tundra   
4  1GT220CG8CZ231238      2012        GMC     Sierra   

                                      Series              FuelTypePrimary  \
0                                       1500                     Gasoline   
1                                    1/2 Ton  Flexible Fuel Vehicle (FFV)   
2                                       1500                     Gasoline   
3  UPK51L/USK51L/USK52L/UPK56L/USK56L/USK57L                     Gasoline   
4                                       2500                     Gasoline   

     Trim              DriveType EngineCylinders DisplacementL  \
0     SLT   RWD/Rear-Wheel Drive               8           5.3   
1      LT                    4x2             